# 83 — Verify Genesis / Vita DAISY / Evaluator six-file FCO/FCG comparison

**Source:** frozen 32-item ECA input dataset previously used by cell-akamai PR #7. **Not recovered historical Genesis or Vita DAISY execution logs.** This successor constructs three *role-specific local fixtures* and their three FCO/FCG projections. The Genesis and Vita fixtures are prompts, **not model outputs**. Evaluator truths are computed by the independent ECA rule. Results do not support antigentic learning or scientific improvement.

This notebook is offline; no model requests or cluster modification.

In [ ]:
import sys,json
from pathlib import Path
HERE=Path.cwd().resolve()
PACKAGE=next((p for p in (HERE,*HERE.parents) if (p/'six_dataset_fco_fcg.py').is_file()),None)
assert PACKAGE, 'Unzip successor archive and run notebook from its notebooks directory'
sys.path.insert(0,str(PACKAGE))
import six_dataset_fco_fcg as sd
result=sd.validate()
print('STATUS',result['status'])
print('SOURCE',result['raw_source_sha256'])
print('LOCAL SIX-FILE MERKLE',result['local_file_merkle']['root_sha256'])
print('PROJECT_MMR',result['project_mmr'])
for kind,x in result['summary'].items():print(kind,x)

## Original structured vs FCO/FCG representation

FCO files preserve full row payloads plus source and occurrence IDs. FCG files record typed provenance edges. **No compression benefit is assumed.** The file sizes and round-trip correctness checks are local computations. The evaluator dataset is held back from model prompts by convention in this package; for production use, enforce ACL isolation.

In [ ]:
from IPython.display import display
import pandas as pd
rows=[]
for lane,s in result['summary'].items():
 rows.append(dict(dataset=lane,items=s['row_count'],raw_bytes=s['raw_bytes'],fco_bytes=s['fco_bytes'],fcg_bytes=s['fcg_bytes'],
                 combined_over_raw=(s['fco_bytes']+s['fcg_bytes'])/s['raw_bytes'],roundtrip=s['reconstruction_exact']))
display(pd.DataFrame(rows))

## Tamper demonstration (offline)

This cell operates on an **in-memory altered row** and does not mutate disk artifacts. It confirms content digest changes; passing hashes alone does not prove the ECA oracle is biologically meaningful.

In [ ]:
original=sd.records(sd.ORIGINAL/'genesis.jsonl')[0]
altered=dict(original);altered['model_answer']='TAMPER'
print('Mutation changes content digest:',sd.sh(sd.encode(original))!=sd.sh(sd.encode(altered)))
assert sd.sh(sd.encode(original))!=sd.sh(sd.encode(altered))
assert all(a['prompt']==o['payload']['prompt'] for _,paired in sd.compare_inputs() for a,o in paired)
print('PASS: raw payload recovered from FCO; model-visible prompts identical for both paths')